In [ ]:
# ==========================================
# NIFTY50 HYBRID PROPHET + ML MODEL
# Direction Classes: -1 (DOWN), 1 (UP)
# ==========================================

import pandas as pd
import numpy as np
from prophet import Prophet
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_csv("NIFTY 50_Historical_PR_01012025to31122025.csv")

df['Date'] = pd.to_datetime(df['Date'], format="%d %b %Y")

df = df.sort_values("Date").reset_index(drop=True)

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================

# returns
df['Return'] = df['Close'].pct_change()

# log returns
df['LogReturn'] = np.log(df['Close']/df['Close'].shift(1))

# volatility
df['Volatility_5'] = df['Return'].rolling(5).std()
df['Volatility_10'] = df['Return'].rolling(10).std()

# moving averages
df['MA5'] = df['Close'].rolling(5).mean()
df['MA10'] = df['Close'].rolling(10).mean()
df['MA20'] = df['Close'].rolling(20).mean()

# momentum
df['Momentum_5'] = df['Close'] - df['Close'].shift(5)

# intraday structure
df['Range'] = df['High'] - df['Low']
df['Body'] = df['Close'] - df['Open']

# RSI
delta = df['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
df['RSI'] = 100 - (100/(1+rs))

# ==========================================
# 3. TARGET LABEL
# ==========================================

df['Direction'] = np.where(df['Close'].shift(-1) > df['Close'], 1, -1)

# Remap -1 to 0 for XGBoost Classifier
df['Direction'] = df['Direction'].replace(-1, 0)

df = df.dropna().reset_index(drop=True)

# ==========================================
# 4. PROPHET MODEL (TREND EXTRACTION)
# ==========================================

prophet_df = df[['Date','Close']].rename(columns={
    'Date':'ds',
    'Close':'y'
})

model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False,
    changepoint_prior_scale=0.05
)

model.fit(prophet_df)

future = prophet_df[['ds']]
forecast = model.predict(future)

# ==========================================
# 5. PROPHET RESIDUALS
# ==========================================

df['Prophet_Pred'] = forecast['yhat']
df['Residual'] = df['Close'] - df['Prophet_Pred']

# ==========================================
# 6. ML FEATURE SET
# ==========================================

features = [
'Return',
'LogReturn',
'Volatility_5',
'Volatility_10',
'MA5',
'MA10',
'MA20',
'Momentum_5',
'Range',
'Body',
'RSI',
'Residual'
]

X = df[features]
y = df['Direction']

# ==========================================
# 7. TRAIN TEST SPLIT
# ==========================================

split = int(len(df)*0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# ==========================================
# 8. SCALING
# ==========================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# 9. ML MODEL
# ==========================================

model_ml = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss'
)

model_ml.fit(X_train, y_train)

# ==========================================
# 10. PREDICTION
# ==========================================

pred = model_ml.predict(X_test)

# ==========================================
# 11. EVALUATION
# ==========================================

print("Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report\n")
print(classification_report(y_test, pred))

Accuracy: 0.6521739130434783

Classification Report

              precision    recall  f1-score   support

           0       0.73      0.62      0.67        26
           1       0.58      0.70      0.64        20

    accuracy                           0.65        46
   macro avg       0.66      0.66      0.65        46
weighted avg       0.66      0.65      0.65        46

